## 1. Load Data and Rebuild Temporal Splits

### Goal
Load the existing processed Parquet dataset and recreate our training, validation, and test sets.

### Split strategy
- **Fit:** January–August 2024
- **Validation:** September 2024
- **Test:** October–December 2024

We use dates rather than a random split to better reflect how the model will perform on future flights.

The test set remains untouched during tuning.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Locate the existing processed dataset
data_path = Path("../data/processed/flights_2024.parquet")

if not data_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {data_path.resolve()}\n"
        "Check the notebook's location and update data_path if needed."
    )

df = pd.read_parquet(data_path)
df["FlightDate"] = pd.to_datetime(df["FlightDate"])

# Recreate chronological splits
fit_df = df[df["FlightDate"].dt.month.between(1, 8)].copy()
val_df = df[df["FlightDate"].dt.month == 9].copy()
test_df = df[df["FlightDate"].dt.month.between(10, 12)].copy()

print(f"Dataset shape: {df.shape}")
print(f"Fit:        {fit_df.shape}")
print(f"Validation: {val_df.shape}")
print(f"Test:       {test_df.shape}")

print("\nDelay rate (DepDel15):")
print(f"Fit:        {fit_df['DepDel15'].mean():.2%}")
print(f"Validation: {val_df['DepDel15'].mean():.2%}")
print(f"Test:       {test_df['DepDel15'].mean():.2%}")

Dataset shape: (6986091, 18)
Fit:        (4638058, 18)
Validation: (579201, 18)
Test:       (1768832, 18)

Delay rate (DepDel15):
Fit:        22.93%
Validation: 15.08%
Test:       16.53%


## 2. Rebuild Features and Train the Baseline

### Goal
Recreate the 13 pre-departure features, align categorical values between fit and validation data, and train the original XGBoost baseline.

### Why this matters
A fresh notebook has no trained model or predictions. We must recreate them before calculating PR-AUC.

### Leakage check
We predict `DepDel15` using only information available before departure. Actual departure times, delays, arrival information, and delay causes must not enter the features.

We will record training time and process memory. Validation will guide early stopping; the test set will not be used.

In [2]:
import os
import time
import psutil
import xgboost as xgb

from sklearn.metrics import average_precision_score

features = [
    "FlightMonth",
    "FlightDay",
    "FlightDayOfWeek",
    "DepHour",
    "DepMinute",
    "DepTimeBlock",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "Route",
    "Distance",
    "ScheduledDuration",
    "DistancePerMinute",
]

target = "DepDel15"
categorical_cols = [
    "DepTimeBlock",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "Route",
]

# Confirm required columns exist
missing = [c for c in features + [target] if c not in df.columns]
assert not missing, f"Missing required columns: {missing}"

# Build fit and validation matrices
X_fit = fit_df[features].copy()
y_fit = fit_df[target].astype("int8")

X_val = val_df[features].copy()
y_val = val_df[target].astype("int8")

# Normalize the single midnight edge case, if present
for X, split_df in [(X_fit, fit_df), (X_val, val_df)]:
    midnight = split_df["CRSDepTime"].eq(2400)
    X.loc[midnight, "DepHour"] = 0
    X.loc[midnight, "DepMinute"] = 0
    X.loc[midnight, "DepTimeBlock"] = "Night"

# Align categories to fit data; unseen validation categories become missing
for col in categorical_cols:
    X_fit[col] = X_fit[col].astype("category")
    categories = X_fit[col].cat.categories
    X_val[col] = pd.Categorical(X_val[col], categories=categories)

# Confirm the feature list contains no known leakage columns
forbidden = {
    "DepTime", "DepDelay", "ArrTime", "ArrDelay",
    "TaxiOut", "WheelsOff", "WheelsOn",
    "CarrierDelay", "WeatherDelay", "NASDelay",
    "SecurityDelay", "LateAircraftDelay",
}
assert not (set(features) & forbidden), "Potential target leakage!"

# Train the original baseline
process = psutil.Process(os.getpid())
memory_before_mb = process.memory_info().rss / (1024 ** 2)
start = time.perf_counter()

model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    enable_categorical=True,
    n_estimators=1500,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=5,
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_fit,
    y_fit,
    eval_set=[(X_val, y_val)],
    verbose=100,
)

train_seconds = time.perf_counter() - start
memory_after_mb = process.memory_info().rss / (1024 ** 2)

val_scores = model.predict_proba(X_val)[:, 1]
baseline_val_pr_auc = average_precision_score(y_val, val_scores)

print(f"\nBaseline validation PR-AUC: {baseline_val_pr_auc:.4f}")
print(f"Best iteration: {model.best_iteration}")
print(f"Training time: {train_seconds / 60:.2f} minutes")
print(f"Memory before: {memory_before_mb:.1f} MB")
print(f"Memory after:  {memory_after_mb:.1f} MB")
print(f"Memory change: {memory_after_mb - memory_before_mb:+.1f} MB")

[0]	validation_0-aucpr:0.23501
[72]	validation_0-aucpr:0.23823

Baseline validation PR-AUC: 0.2442
Best iteration: 22
Training time: 0.97 minutes
Memory before: 980.0 MB
Memory after:  1062.1 MB
Memory change: +82.0 MB


## 3. Tune Model Complexity

### Goal
Find a simpler XGBoost model that performs at least as well on unseen validation data while reducing overfitting.

### Approach
- Train 8 configurations on a fixed 250,000-row sample.
- Test shallower trees, stronger regularization, and different learning rates.
- Use September validation PR-AUC and early stopping to compare configurations.
- Calculate the training–validation PR-AUC gap as an overfitting indicator.

A smaller gap is generally encouraging, but validation PR-AUC remains the main selection metric. We will retrain the selected configuration on the full fit set before final evaluation.

In [3]:
import time
import psutil
import os
import pandas as pd
import xgboost as xgb
from sklearn.metrics import average_precision_score

process = psutil.Process(os.getpid())
memory_before_mb = process.memory_info().rss / (1024 ** 2)
start = time.perf_counter()

# Use the same sample for every configuration
sampled_X_fit = X_fit.sample(n=250_000, random_state=42)
sampled_y_fit = y_fit.loc[sampled_X_fit.index]

# Baseline gap measured on that same sample
baseline_train_scores = model.predict_proba(sampled_X_fit)[:, 1]
baseline_train_pr_auc = average_precision_score(
    sampled_y_fit, baseline_train_scores
)
baseline_gap = baseline_train_pr_auc - baseline_val_pr_auc

configs = [
    {"max_depth": 3, "min_child_weight": 20, "learning_rate": 0.03, "reg_lambda": 10},
    {"max_depth": 3, "min_child_weight": 50, "learning_rate": 0.03, "reg_lambda": 20},
    {"max_depth": 4, "min_child_weight": 20, "learning_rate": 0.03, "reg_lambda": 10},
    {"max_depth": 4, "min_child_weight": 50, "learning_rate": 0.03, "reg_lambda": 20},
    {"max_depth": 3, "min_child_weight": 20, "learning_rate": 0.02, "reg_lambda": 20},
    {"max_depth": 4, "min_child_weight": 20, "learning_rate": 0.02, "reg_lambda": 30},
    {"max_depth": 3, "min_child_weight": 50, "learning_rate": 0.05, "reg_lambda": 30},
    {"max_depth": 4, "min_child_weight": 50, "learning_rate": 0.02, "reg_lambda": 50},
]

results = []
candidate_models = []

for i, params in enumerate(configs, start=1):
    candidate = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        enable_categorical=True,
        n_estimators=500,
        subsample=0.8,
        colsample_bytree=0.8,
        early_stopping_rounds=30,
        random_state=42,
        n_jobs=-1,
        **params,
    )

    candidate.fit(
        sampled_X_fit,
        sampled_y_fit,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    train_scores = candidate.predict_proba(sampled_X_fit)[:, 1]
    val_scores_candidate = candidate.predict_proba(X_val)[:, 1]

    train_ap = average_precision_score(sampled_y_fit, train_scores)
    val_ap = average_precision_score(y_val, val_scores_candidate)

    results.append({
        **params,
        "best_iteration": candidate.best_iteration,
        "train_pr_auc": train_ap,
        "val_pr_auc": val_ap,
        "train_val_gap": train_ap - val_ap,
    })
    candidate_models.append(candidate)

    print(
        f"Run {i}/8 | validation PR-AUC: {val_ap:.4f} | "
        f"best iteration: {candidate.best_iteration}"
    )

tuning_results = pd.DataFrame(results).sort_values(
    "val_pr_auc", ascending=False
).reset_index(drop=True)

elapsed_seconds = time.perf_counter() - start
memory_after_mb = process.memory_info().rss / (1024 ** 2)

print(f"\nBaseline validation PR-AUC: {baseline_val_pr_auc:.4f}")
print(f"Baseline sampled training PR-AUC: {baseline_train_pr_auc:.4f}")
print(f"Baseline sampled train-validation gap: {baseline_gap:.4f}")

display(tuning_results.round(4))

print(f"\nSweep time: {elapsed_seconds / 60:.2f} minutes")
print(f"Memory before: {memory_before_mb:.1f} MB")
print(f"Memory after: {memory_after_mb:.1f} MB")
print(f"Memory change: {memory_after_mb - memory_before_mb:+.1f} MB")

Run 1/8 | validation PR-AUC: 0.2367 | best iteration: 146
Run 2/8 | validation PR-AUC: 0.2358 | best iteration: 60
Run 3/8 | validation PR-AUC: 0.2354 | best iteration: 14
Run 4/8 | validation PR-AUC: 0.2369 | best iteration: 115
Run 5/8 | validation PR-AUC: 0.2369 | best iteration: 184
Run 6/8 | validation PR-AUC: 0.2368 | best iteration: 54
Run 7/8 | validation PR-AUC: 0.2369 | best iteration: 67
Run 8/8 | validation PR-AUC: 0.2360 | best iteration: 27

Baseline validation PR-AUC: 0.2442
Baseline sampled training PR-AUC: 0.4331
Baseline sampled train-validation gap: 0.1889


,max_depth,min_child_weight,learning_rate,reg_lambda,best_iteration,train_pr_auc,val_pr_auc,train_val_gap
0,3,50,0.05,30,67,0.4165,0.2369,0.1796
1,3,20,0.02,20,184,0.4189,0.2369,0.1820
2,4,50,0.03,20,115,0.4332,0.2369,0.1963
3,4,20,0.02,30,54,0.4088,0.2368,0.1720
4,3,20,0.03,10,146,0.4245,0.2367,0.1878
5,4,50,0.02,50,27,0.3978,0.2360,0.1618
6,3,50,0.03,20,60,0.4062,0.2358,0.1704
7,4,20,0.03,10,14,0.4004,0.2354,0.1650



Sweep time: 3.38 minutes
Memory before: 885.7 MB
Memory after: 976.4 MB
Memory change: +90.7 MB


In [4]:
display(tuning_results.round(4))

,max_depth,min_child_weight,learning_rate,reg_lambda,best_iteration,train_pr_auc,val_pr_auc,train_val_gap
0,3,50,0.05,30,67,0.4165,0.2369,0.1796
1,3,20,0.02,20,184,0.4189,0.2369,0.1820
2,4,50,0.03,20,115,0.4332,0.2369,0.1963
3,4,20,0.02,30,54,0.4088,0.2368,0.1720
4,3,20,0.03,10,146,0.4245,0.2367,0.1878
5,4,50,0.02,50,27,0.3978,0.2360,0.1618
6,3,50,0.03,20,60,0.4062,0.2358,0.1704
7,4,20,0.03,10,14,0.4004,0.2354,0.1650


## 4. Refit the Selected Model

### Goal
Train the best candidate on the full fit set and select its classification threshold using validation data.

### Why this configuration?
It achieved the highest validation PR-AUC among the 8 tuning candidates. The alternative with the smallest gap performed worse on validation.

### Evaluation rules
- Fit on January–August.
- Use September for early stopping and threshold selection.
- Do not evaluate October–December until the final evaluation step.

We will compare the new validation score and sampled training–validation gap against the baseline.

In [5]:
from sklearn.metrics import average_precision_score, precision_recall_curve
import time
import psutil
import os
import xgboost as xgb

process = psutil.Process(os.getpid())
memory_before_mb = process.memory_info().rss / (1024 ** 2)
start = time.perf_counter()

final_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    enable_categorical=True,
    n_estimators=500,
    learning_rate=0.05,
    max_depth=3,
    min_child_weight=50,
    reg_lambda=30,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1,
)

final_model.fit(
    X_fit,
    y_fit,
    eval_set=[(X_val, y_val)],
    verbose=100,
)

train_seconds = time.perf_counter() - start
memory_after_mb = process.memory_info().rss / (1024 ** 2)

# Validation evaluation only
val_scores_final = final_model.predict_proba(X_val)[:, 1]
val_pr_auc_final = average_precision_score(y_val, val_scores_final)

# Compare training and validation using the same fixed fit sample
train_scores_final = final_model.predict_proba(sampled_X_fit)[:, 1]
train_pr_auc_final = average_precision_score(
    sampled_y_fit, train_scores_final
)
gap_final = train_pr_auc_final - val_pr_auc_final

# Select threshold by maximizing F1 on validation
precision, recall, thresholds = precision_recall_curve(
    y_val, val_scores_final
)
f1_scores = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-12)
)
best_idx = f1_scores.argmax()
best_threshold = thresholds[best_idx]

print(f"\nFinal best iteration: {final_model.best_iteration}")
print(f"Final validation PR-AUC: {val_pr_auc_final:.4f}")
print(f"Validation-selected threshold: {best_threshold:.4f}")
print(f"Validation precision at threshold: {precision[best_idx]:.4f}")
print(f"Validation recall at threshold: {recall[best_idx]:.4f}")
print(f"Validation F1 at threshold: {f1_scores[best_idx]:.4f}")

print(f"\nBaseline sampled train PR-AUC: {baseline_train_pr_auc:.4f}")
print(f"Baseline validation PR-AUC:    {baseline_val_pr_auc:.4f}")
print(f"Baseline sampled gap:          {baseline_gap:.4f}")
print(f"Final sampled train PR-AUC:    {train_pr_auc_final:.4f}")
print(f"Final validation PR-AUC:       {val_pr_auc_final:.4f}")
print(f"Final sampled gap:             {gap_final:.4f}")

print(f"\nFull refit time: {train_seconds / 60:.2f} minutes")
print(f"Memory before: {memory_before_mb:.1f} MB")
print(f"Memory after:  {memory_after_mb:.1f} MB")
print(f"Memory change: {memory_after_mb - memory_before_mb:+.1f} MB")

[0]	validation_0-aucpr:0.21648
[100]	validation_0-aucpr:0.24046
[131]	validation_0-aucpr:0.23753

Final best iteration: 101
Final validation PR-AUC: 0.2407
Validation-selected threshold: 0.2716
Validation precision at threshold: 0.2287
Validation recall at threshold: 0.5664
Validation F1 at threshold: 0.3258

Baseline sampled train PR-AUC: 0.4331
Baseline validation PR-AUC:    0.2442
Baseline sampled gap:          0.1889
Final sampled train PR-AUC:    0.4082
Final validation PR-AUC:       0.2407
Final sampled gap:             0.1675

Full refit time: 1.12 minutes
Memory before: 978.5 MB
Memory after:  1052.4 MB
Memory change: +73.9 MB


## 5. Final Test Evaluation

### Goal
Evaluate the selected model on October–December 2024, after completing model selection using validation data.

### Metrics
- PR-AUC and lift over the test-set delay prevalence
- ROC-AUC
- Precision, recall, and F1 using the threshold selected on validation

The test set is used only for this final evaluation, not for tuning or threshold selection.

In [6]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
)

# Prepare test features using the fit-set category definitions
X_test = test_df[features].copy()
y_test = test_df[target].astype("int8")

midnight_test = test_df["CRSDepTime"].eq(2400)
X_test.loc[midnight_test, "DepHour"] = 0
X_test.loc[midnight_test, "DepMinute"] = 0
X_test.loc[midnight_test, "DepTimeBlock"] = "Night"

for col in categorical_cols:
    X_test[col] = pd.Categorical(
        X_test[col],
        categories=X_fit[col].cat.categories,
    )

# One final test evaluation
test_scores_final = final_model.predict_proba(X_test)[:, 1]
test_pr_auc_final = average_precision_score(y_test, test_scores_final)
test_prevalence = y_test.mean()
test_lift = test_pr_auc_final / test_prevalence
test_roc_auc = roc_auc_score(y_test, test_scores_final)

test_predictions = (test_scores_final >= best_threshold).astype("int8")

print(f"Test prevalence: {test_prevalence:.4f}")
print(f"Test PR-AUC: {test_pr_auc_final:.4f}")
print(f"Test PR-AUC lift: {test_lift:.3f}x")
print(f"Test ROC-AUC: {test_roc_auc:.4f}")
print(f"Test precision: {precision_score(y_test, test_predictions):.4f}")
print(f"Test recall: {recall_score(y_test, test_predictions):.4f}")
print(f"Test F1: {f1_score(y_test, test_predictions):.4f}")
print(f"Validation-selected threshold: {best_threshold:.4f}")

Test prevalence: 0.1653
Test PR-AUC: 0.2443
Test PR-AUC lift: 1.478x
Test ROC-AUC: 0.6389
Test precision: 0.2359
Test recall: 0.5342
Test F1: 0.3273
Validation-selected threshold: 0.2716


## 6. Monthly Test Breakdown

### Goal
Check whether model performance is consistent across October, November, and December.

For each month, calculate PR-AUC, delay prevalence, and PR-AUC lift over that month's prevalence. This is a diagnostic breakdown, not another tuning step.

In [7]:
monthly_results = []

test_months = test_df["FlightDate"].dt.month.to_numpy()
y_test_array = np.asarray(y_test)

for month in [10, 11, 12]:
    mask = test_months == month

    month_y = y_test_array[mask]
    month_scores = test_scores_final[mask]

    prevalence = month_y.mean()
    pr_auc = average_precision_score(month_y, month_scores)

    monthly_results.append({
        "month": {10: "October", 11: "November", 12: "December"}[month],
        "rows": int(mask.sum()),
        "delay_prevalence": prevalence,
        "pr_auc": pr_auc,
        "lift": pr_auc / prevalence,
    })

monthly_results = pd.DataFrame(monthly_results)
display(monthly_results.round(4))

,month,rows,delay_prevalence,pr_auc,lift
0,October,609295,0.1361,0.2141,1.5736
1,November,572872,0.1441,0.2141,1.4851
2,December,586665,0.2165,0.3057,1.4119


## Baseline Stability Check

### Goal
Measure how much validation PR-AUC changes across three random seeds while keeping the dataset, chronological split, features, and baseline hyperparameters unchanged.

### Why
A tiny score difference may be caused by training randomness rather than a genuine model improvement.

We will use validation PR-AUC only. The test set will not be used for this comparison.

In [8]:
import time
import pandas as pd
import xgboost as xgb
from sklearn.metrics import average_precision_score

seeds = [21, 42, 84]
seed_results = []

for seed in seeds:
    model_seed = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        enable_categorical=True,
        n_estimators=1500,
        learning_rate=0.05,
        max_depth=8,
        min_child_weight=10,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=5,
        early_stopping_rounds=50,
        random_state=seed,
        n_jobs=-1,
    )

    start = time.perf_counter()

    model_seed.fit(
        X_fit,
        y_fit,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    val_scores_seed = model_seed.predict_proba(X_val)[:, 1]
    val_ap = average_precision_score(y_val, val_scores_seed)

    seed_results.append({
        "seed": seed,
        "best_iteration": model_seed.best_iteration,
        "validation_pr_auc": val_ap,
        "training_minutes": (time.perf_counter() - start) / 60,
    })

seed_results = pd.DataFrame(seed_results)
display(seed_results.round(4))

print(f"Mean validation PR-AUC: {seed_results['validation_pr_auc'].mean():.4f}")
print(f"Score range: {seed_results['validation_pr_auc'].min():.4f}"
      f"–{seed_results['validation_pr_auc'].max():.4f}")
print(f"Range width: {seed_results['validation_pr_auc'].max() - seed_results['validation_pr_auc'].min():.4f}")

,seed,best_iteration,validation_pr_auc,training_minutes
0,21,38,0.2448,1.1790
1,42,22,0.2442,1.0996
2,84,5,0.2445,1.0480


Mean validation PR-AUC: 0.2445
Score range: 0.2442–0.2448
Range width: 0.0006


## Primary Model Decision

**Chosen model:** Original XGBoost baseline (`max_depth=8`, `min_child_weight=10`, `learning_rate=0.05`, `reg_lambda=5`).

**Reason:** Baseline validation PR-AUC averaged 0.2445 across three seeds, with a 0.0006 range. The tuned model's 0.0006 test PR-AUC advantage is too small to establish a reliable improvement, and its validation PR-AUC was lower (0.2407 vs. 0.2442). Keep the baseline as primary and retain the tuned model as an experiment.

---

In [9]:
# Weather pilot: August 2024, using the 10 busiest origin airports

pilot_month = df.loc[df["FlightDate"].dt.month == 8].copy()

airport_counts = (
    pilot_month["Origin"]
    .value_counts()
    .head(10)
    .rename_axis("Origin")
    .reset_index(name="flight_count")
)

print(f"August 2024 flight rows: {len(pilot_month):,}")
display(airport_counts)

August 2024 flight rows: 606,942


,Origin,flight_count
0,ATL,29187
1,DFW,27632
2,DEN,27148
3,ORD,25167
4,CLT,18559
5,LAX,17049
6,LAS,15963
7,SEA,15861
8,PHX,15315
9,LGA,13143


In [10]:
# Inspect fields needed to calculate the T-2-hour weather cutoff

time_columns = [
    c for c in df.columns
    if any(term in c.lower() for term in ["date", "time", "origin"])
]

print("Potential date/time columns:")
print(time_columns)

display(pilot_month[
    ["FlightDate", "Origin", "CRSDepTime"]
].head(10))

print("\nCRSDepTime examples and data types:")
print(pilot_month["CRSDepTime"].dtype)
print(pilot_month["CRSDepTime"].value_counts().head(10))

Potential date/time columns:
['FlightDate', 'Origin', 'CRSDepTime', 'CRSElapsedTime', 'DepTimeBlock']


,FlightDate,Origin,CRSDepTime
5799948,2024-08-09,PHL,1051
5799949,2024-08-10,PHL,1051
5799950,2024-08-11,PHL,1051
5799951,2024-08-12,PHL,1051
5799952,2024-08-13,PHL,1051
5799953,2024-08-14,PHL,1051
5799954,2024-08-15,PHL,1051
5799955,2024-08-16,PHL,1051
5799956,2024-08-17,PHL,1051
5799957,2024-08-18,PHL,1051



CRSDepTime examples and data types:
int16
CRSDepTime
600     13597
700     10506
800      5444
900      4089
630      3705
830      3672
1000     3661
715      3409
1100     3213
710      3126
Name: count, dtype: int64


In [12]:
# Check scheduled-time fields and pilot airport coverage

print("Scheduled departure fields:")
print([
    c for c in pilot_month.columns
    if c in ["FlightDate", "CRSDepTime", "CRSDepDate", "Origin"]
])

print("\nPilot airports:")
display(airport_counts)

print("\nSample scheduled dates and times:")
display(
    pilot_month[["FlightDate", "CRSDepTime", "Origin"]]
    .head(10)
)

Scheduled departure fields:
['FlightDate', 'Origin', 'CRSDepTime']

Pilot airports:


,Origin,flight_count
0,ATL,29187
1,DFW,27632
2,DEN,27148
3,ORD,25167
4,CLT,18559
5,LAX,17049
6,LAS,15963
7,SEA,15861
8,PHX,15315
9,LGA,13143



Sample scheduled dates and times:


,FlightDate,CRSDepTime,Origin
5799948,2024-08-09,1051,PHL
5799949,2024-08-10,1051,PHL
5799950,2024-08-11,1051,PHL
5799951,2024-08-12,1051,PHL
5799952,2024-08-13,1051,PHL
5799953,2024-08-14,1051,PHL
5799954,2024-08-15,1051,PHL
5799955,2024-08-16,1051,PHL
5799956,2024-08-17,1051,PHL
5799957,2024-08-18,1051,PHL


In [13]:
# Restrict the pilot to the 10 busiest August origin airports
pilot_airports = airport_counts["Origin"].tolist()

pilot_flights = pilot_month[
    pilot_month["Origin"].isin(pilot_airports)
].copy()

# Check HHMM values before converting to timestamps
hhmm = pilot_flights["CRSDepTime"]

invalid_time = (
    (hhmm < 0)
    | (hhmm > 2400)
    | ((hhmm % 100) >= 60)
    | ((hhmm == 2400) & (hhmm % 100 != 0))
)

print(f"Pilot flight rows: {len(pilot_flights):,}")
print(f"Airports included: {pilot_flights['Origin'].nunique()}")
print(f"Invalid scheduled-time rows: {invalid_time.sum():,}")
print(f"2400 (midnight) rows: {(hhmm == 2400).sum():,}")

display(
    pilot_flights.loc[invalid_time, ["FlightDate", "Origin", "CRSDepTime"]]
    .head(10)
)

Pilot flight rows: 205,024
Airports included: 10
Invalid scheduled-time rows: 0
2400 (midnight) rows: 0


,FlightDate,Origin,CRSDepTime


In [14]:
# Airport time zones for the 10-airport weather pilot

airport_timezones = {
    "ATL": "America/New_York",
    "DFW": "America/Chicago",
    "DEN": "America/Denver",
    "ORD": "America/Chicago",
    "CLT": "America/New_York",
    "LAX": "America/Los_Angeles",
    "LAS": "America/Los_Angeles",
    "SEA": "America/Los_Angeles",
    "PHX": "America/Phoenix",
    "LGA": "America/New_York",
}

assert set(pilot_airports) == set(airport_timezones), (
    "Time-zone mapping does not match pilot airports"
)

display(pd.DataFrame([
    {"Origin": airport, "timezone": timezone}
    for airport, timezone in airport_timezones.items()
]))

,Origin,timezone
0,ATL,America/New_York
1,DFW,America/Chicago
2,DEN,America/Denver
3,ORD,America/Chicago
4,CLT,America/New_York
5,LAX,America/Los_Angeles
6,LAS,America/Los_Angeles
7,SEA,America/Los_Angeles
8,PHX,America/Phoenix
9,LGA,America/New_York


In [16]:
# Convert scheduled local departure times to UTC, then apply T-2h cutoff

pilot_flights["scheduled_departure_local"] = pd.NaT
pilot_flights["scheduled_departure_utc"] = pd.NaT

for airport, timezone in airport_timezones.items():
    mask = pilot_flights["Origin"].eq(airport)

    hhmm = pilot_flights.loc[mask, "CRSDepTime"].astype(int)
    hours = hhmm // 100
    minutes = hhmm % 100

    local_naive = (
        pilot_flights.loc[mask, "FlightDate"].dt.normalize()
        + pd.to_timedelta(hours, unit="h")
        + pd.to_timedelta(minutes, unit="m")
    )

    local_aware = local_naive.dt.tz_localize(
        timezone,
        ambiguous="NaT",
        nonexistent="NaT",
    )

    pilot_flights.loc[mask, "scheduled_departure_local"] = (
        local_aware.astype("datetime64[ns, UTC]")
    )
    pilot_flights.loc[mask, "scheduled_departure_utc"] = (
        local_aware.dt.tz_convert("UTC")
    )

pilot_flights["weather_cutoff_utc"] = (
    pilot_flights["scheduled_departure_utc"]
    - pd.Timedelta(hours=2)
)

print("Rows:", len(pilot_flights))
print("Missing UTC departure:", pilot_flights["scheduled_departure_utc"].isna().sum())
print("Missing weather cutoff:", pilot_flights["weather_cutoff_utc"].isna().sum())

display(
    pilot_flights[
        ["FlightDate", "Origin", "CRSDepTime",
         "scheduled_departure_utc", "weather_cutoff_utc"]
    ].head(10)
)

TypeError: Invalid value '<DatetimeArray>
['2024-08-01 19:21:00+00:00', '2024-08-02 19:21:00+00:00',
 '2024-08-03 19:21:00+00:00', '2024-08-05 19:21:00+00:00',
 '2024-08-01 09:00:00+00:00', '2024-08-02 09:00:00+00:00',
 '2024-08-05 09:00:00+00:00', '2024-08-06 09:10:00+00:00',
 '2024-08-07 09:10:00+00:00', '2024-08-08 09:10:00+00:00',
 ...
 '2024-08-07 21:45:00+00:00', '2024-08-07 14:20:00+00:00',
 '2024-08-07 09:30:00+00:00', '2024-08-07 13:32:00+00:00',
 '2024-08-07 20:20:00+00:00', '2024-08-06 23:25:00+00:00',
 '2024-08-06 15:12:00+00:00', '2024-08-06 16:00:00+00:00',
 '2024-08-06 16:50:00+00:00', '2024-08-06 13:35:00+00:00']
Length: 29187, dtype: datetime64[ns, UTC]' for dtype 'datetime64[ns]'

In [17]:
# Build scheduled departure timestamps and T-2h weather cutoffs in UTC

utc_departures = pd.Series(
    pd.NaT,
    index=pilot_flights.index,
    dtype="datetime64[ns, UTC]",
)

for airport, timezone in airport_timezones.items():
    mask = pilot_flights["Origin"].eq(airport)

    hhmm = pilot_flights.loc[mask, "CRSDepTime"].astype(int)

    local_naive = (
        pilot_flights.loc[mask, "FlightDate"].dt.normalize()
        + pd.to_timedelta(hhmm // 100, unit="h")
        + pd.to_timedelta(hhmm % 100, unit="m")
    )

    local_aware = local_naive.dt.tz_localize(
        timezone,
        ambiguous="NaT",
        nonexistent="NaT",
    )

    utc_departures.loc[mask] = local_aware.dt.tz_convert("UTC")

pilot_flights["scheduled_departure_utc"] = utc_departures
pilot_flights["weather_cutoff_utc"] = (
    pilot_flights["scheduled_departure_utc"] - pd.Timedelta(hours=2)
)

print(f"Pilot rows: {len(pilot_flights):,}")
print(f"Missing UTC departure: {utc_departures.isna().sum():,}")
print(f"Missing weather cutoff: {pilot_flights['weather_cutoff_utc'].isna().sum():,}")

display(
    pilot_flights[
        ["FlightDate", "Origin", "CRSDepTime",
         "scheduled_departure_utc", "weather_cutoff_utc"]
    ].head(10)
)

Pilot rows: 205,024
Missing UTC departure: 0
Missing weather cutoff: 0


,FlightDate,Origin,CRSDepTime,scheduled_departure_utc,weather_cutoff_utc
5799971,2024-08-06,LAX,1030,2024-08-06 17:30:00+00:00,2024-08-06 15:30:00+00:00
5799972,2024-08-07,LAX,1030,2024-08-07 17:30:00+00:00,2024-08-07 15:30:00+00:00
5799973,2024-08-08,LAX,1030,2024-08-08 17:30:00+00:00,2024-08-08 15:30:00+00:00
5799974,2024-08-09,LAX,1030,2024-08-09 17:30:00+00:00,2024-08-09 15:30:00+00:00
5799975,2024-08-10,LAX,1030,2024-08-10 17:30:00+00:00,2024-08-10 15:30:00+00:00
5799976,2024-08-11,LAX,1030,2024-08-11 17:30:00+00:00,2024-08-11 15:30:00+00:00
5799977,2024-08-12,LAX,1030,2024-08-12 17:30:00+00:00,2024-08-12 15:30:00+00:00
5799978,2024-08-13,LAX,1030,2024-08-13 17:30:00+00:00,2024-08-13 15:30:00+00:00
5799979,2024-08-14,LAX,1030,2024-08-14 17:30:00+00:00,2024-08-14 15:30:00+00:00
5799980,2024-08-15,LAX,1030,2024-08-15 17:30:00+00:00,2024-08-15 15:30:00+00:00
